# 00 - Exploracion del dataset


## Aviso inicial

Analisis previo realizado sobre la raiz del proyecto:

- PDF encontrado: `LuisHernandezRodriguez_Datasets Elegidos.docx.pdf`. Describe tres bloques visuales: D-Fire, Open Wildfire Smoke y Cloud/Fog.
- Archivos comprimidos encontrados en `Images/`:
  - `D-Fire.zip`: 21.527 imagenes `.jpg`, 21.527 etiquetas `.txt` YOLO y `data.yaml`.
  - `day_time_wildfire_v2.tar.gz`: 2.191 imagenes `.jpeg` y 2.191 anotaciones Pascal VOC `.xml`.
  - `cloud.tar.gz`: 1.296 imagenes `.jpg` sin anotaciones, adecuadas como negativos.

Antes de ejecutar el analisis, extrae esos comprimidos en:

- `Images/D-Fire.zip` -> `data/raw/dfire/`
- `Images/day_time_wildfire_v2.tar.gz` -> `data/raw/wildfire_smoke/`
- `Images/cloud.tar.gz` -> `data/raw/cloud_fog/`

Las busquedas son recursivas para tolerar las carpetas internas reales (`data/test/images`, `annotations/xmls`, `cloud/`, etc.).


## 1. Comprobacion del entorno


In [ ]:
import platform
import sys

rows = [("Python", sys.version.split()[0]), ("Sistema", platform.platform())]
try:
    import torch
    rows.extend([
        ("PyTorch", torch.__version__),
        ("CUDA disponible", torch.cuda.is_available()),
        ("CUDA runtime PyTorch", torch.version.cuda),
        ("GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No detectada"),
    ])
except Exception as exc:
    rows.append(("PyTorch", f"No disponible: {exc}"))

try:
    import ultralytics
    rows.append(("Ultralytics", ultralytics.__version__))
except Exception as exc:
    rows.append(("Ultralytics", f"No disponible: {exc}"))

pd.DataFrame(rows, columns=["Componente", "Valor"])


## 2. Inspeccion de datasets en raw/


In [ ]:
from pathlib import Path
import json
import random
import shutil
import sys
import time
import warnings
import xml.etree.ElementTree as ET

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATASETS = {
    "dfire": RAW_DIR / "dfire",
    "wildfire_smoke": RAW_DIR / "wildfire_smoke",
    "cloud_fog": RAW_DIR / "cloud_fog",
}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff", ".tif"}
CLASS_NAMES = {0: "fire", 1: "smoke"}
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Project root: {PROJECT_ROOT}")


def find_images(root: Path):
    if not root.exists():
        return []
    return sorted(p for p in root.rglob("*") if p.suffix.lower() in IMAGE_EXTS and not p.name.startswith("._"))


def detect_annotation_formats(root: Path):
    if not root.exists():
        return ["missing"]
    formats = []
    if list(root.rglob("*.txt")):
        formats.append("YOLO .txt or text")
    if list(root.rglob("*.xml")):
        formats.append("Pascal VOC .xml")
    json_files = list(root.rglob("*.json"))
    if json_files:
        coco = False
        for path in json_files[:5]:
            try:
                data = json.loads(path.read_text(encoding="utf-8"))
                coco = coco or {"images", "annotations"}.issubset(data.keys())
            except Exception:
                pass
        formats.append("COCO .json" if coco else "JSON")
    return formats or ["none detected"]


def sibling_yolo_label(image_path: Path):
    candidates = [image_path.with_suffix(".txt")]
    parts = list(image_path.parts)
    if "images" in parts:
        idx = parts.index("images")
        label_parts = parts.copy()
        label_parts[idx] = "labels"
        candidates.append(Path(*label_parts).with_suffix(".txt"))
    return next((p for p in candidates if p.exists()), None)


def parse_yolo_lines(label_path: Path, allowed_classes={0, 1}):
    boxes = []
    if not label_path or not label_path.exists() or label_path.stat().st_size == 0:
        return boxes
    for line in label_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        try:
            cls = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
        except ValueError:
            continue
        if cls in allowed_classes and all(0 <= v <= 1 for v in (x, y, w, h)) and w > 0 and h > 0:
            boxes.append((cls, x, y, w, h))
    return boxes


def yolo_to_xyxy(box, width, height):
    cls, x, y, w, h = box
    x1 = int((x - w / 2) * width)
    y1 = int((y - h / 2) * height)
    x2 = int((x + w / 2) * width)
    y2 = int((y + h / 2) * height)
    return cls, max(0, x1), max(0, y1), min(width - 1, x2), min(height - 1, y2)


def draw_yolo_boxes(ax, image_path: Path, boxes):
    img = Image.open(image_path).convert("RGB")
    width, height = img.size
    ax.imshow(img)
    for box in boxes:
        cls, x1, y1, x2, y2 = yolo_to_xyxy(box, width, height)
        color = "orangered" if cls == 0 else "gray"
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=2, color=color))
        ax.text(x1, max(0, y1 - 4), CLASS_NAMES.get(cls, str(cls)), color="white", fontsize=8,
                bbox=dict(facecolor=color, alpha=0.8, edgecolor="none", pad=1))
    ax.set_title(image_path.name[:32], fontsize=8)
    ax.axis("off")


# ADAPTADO: el D-Fire.zip local declara names: ['smoke', 'fire'].
# El dataset final del proyecto debe ser 0=fire y 1=smoke, asi que se remapea.
DFIRE_CLASS_REMAP = {0: 1, 1: 0}


def parse_dfire_yolo_lines(label_path: Path):
    return [(DFIRE_CLASS_REMAP.get(cls, cls), x, y, w, h) for cls, x, y, w, h in parse_yolo_lines(label_path)]


In [ ]:
summary_rows = []
for name, path in DATASETS.items():
    images = find_images(path)
    summary_rows.append({
        "dataset": name,
        "path": str(path),
        "exists": path.exists(),
        "images": len(images),
        "annotation_formats": ", ".join(detect_annotation_formats(path)),
    })
pd.DataFrame(summary_rows)


In [ ]:
for dataset_name, dataset_path in DATASETS.items():
    images = find_images(dataset_path)
    if not images:
        print(f"[INFO] {dataset_name}: sin imagenes en {dataset_path}")
        continue
    sample = random.sample(images, min(9, len(images)))
    fig, axes = plt.subplots(3, 3, figsize=(12, 10))
    fig.suptitle(f"Ejemplos de {dataset_name}")
    for ax, image_path in zip(axes.ravel(), sample):
        label_path = sibling_yolo_label(image_path)
        boxes = parse_dfire_yolo_lines(label_path) if dataset_name == "dfire" else parse_yolo_lines(label_path)
        draw_yolo_boxes(ax, image_path, boxes)
    for ax in axes.ravel()[len(sample):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


### Distribucion de clases D-Fire


In [ ]:
counts = {"fire": 0, "smoke": 0}
for image_path in find_images(DATASETS["dfire"]):
    for cls, *_ in parse_dfire_yolo_lines(sibling_yolo_label(image_path)):
        counts["fire" if cls == 0 else "smoke"] += 1
pd.DataFrame([counts]).T.rename(columns={0: "bounding_boxes"})


### Resoluciones y tamanos relativos de bounding boxes


In [ ]:
resolution_rows, bbox_rows = [], []
for dataset_name, dataset_path in DATASETS.items():
    for image_path in find_images(dataset_path):
        try:
            with Image.open(image_path) as img:
                w_img, h_img = img.size
            resolution_rows.append({"dataset": dataset_name, "width": w_img, "height": h_img})
            box_reader = parse_dfire_yolo_lines if dataset_name == "dfire" else parse_yolo_lines
            for cls, x, y, w, h in box_reader(sibling_yolo_label(image_path)):
                bbox_rows.append({"dataset": dataset_name, "class": CLASS_NAMES[cls], "relative_area": w * h})
        except Exception as exc:
            print(f"[WARN] No se pudo leer {image_path}: {exc}")

resolutions = pd.DataFrame(resolution_rows)
bboxes = pd.DataFrame(bbox_rows)

if resolutions.empty:
    print("No hay imagenes para analizar.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    resolutions["width"].hist(ax=axes[0], bins=30)
    axes[0].set_title("Anchuras")
    resolutions["height"].hist(ax=axes[1], bins=30)
    axes[1].set_title("Alturas")
    plt.tight_layout()
    plt.show()
    display(resolutions.groupby("dataset")[["width", "height"]].mean().round(1))

if bboxes.empty:
    print("No hay bounding boxes YOLO para analizar tamanos.")
else:
    bboxes["relative_area"].hist(bins=40, figsize=(8, 4))
    plt.title("Area relativa bbox / imagen")
    plt.show()
    display(bboxes.groupby(["dataset", "class"])["relative_area"].describe())


## 3. Resumen estadistico


In [ ]:
rows = []
for dataset_name, dataset_path in DATASETS.items():
    images = find_images(dataset_path)
    ann = 0
    classes = set()
    widths, heights = [], []
    for image_path in images:
        try:
            with Image.open(image_path) as img:
                w, h = img.size
            widths.append(w); heights.append(h)
        except Exception:
            pass
        box_reader = parse_dfire_yolo_lines if dataset_name == "dfire" else parse_yolo_lines
        boxes = box_reader(sibling_yolo_label(image_path))
        ann += len(boxes)
        classes.update(CLASS_NAMES.get(cls, str(cls)) for cls, *_ in boxes)
    rows.append({
        "dataset": dataset_name,
        "imagenes_totales": len(images),
        "anotaciones": ann,
        "clases_presentes": ", ".join(sorted(classes)) if classes else "-",
        "resolucion_media": f"{np.mean(widths):.0f}x{np.mean(heights):.0f}" if widths else "-",
    })
pd.DataFrame(rows)
